Hashing · Study Notes  
Ref: EPI Chapter 9 

KTH  
July 1, 2026

## 1. Hash table fundamentals

A **hash table** is a data structure used to store keys, optionally with
corresponding values. Inserts, deletes, and lookups run in `O(1)` time **on
average**.

**How it works.** Keys are stored in an array. A key is placed in an array
location ("slot") based on its **hash code** — an integer computed from the key
by a **hash function**. If the hash function is chosen well, objects are
distributed uniformly across the array locations.

**Collisions.** If two keys map to the same location, a **collision** occurs.
The standard mechanism for handling collisions is to maintain a **linked list**
of objects at each array location. If the hash function spreads objects well and
computes in `O(1)` time, then on average, lookups, insertions, and deletions have
`O(1 + n/m)` time complexity, where `n` is the number of objects and `m` is the
array length.

**Load factor & rehashing.** If the "load" `n/m` grows large, **rehashing** can
be applied: a new, larger array is allocated and all objects are moved over.
Rehashing costs `O(n + m)`, but if done infrequently (e.g., whenever the number
of entries doubles), its amortized cost is low.

**Hash table vs. sorted array.** A hash table is qualitatively different from a
sorted array — keys need not appear in order, and randomization (the hash
function) plays a central role. Compared to binary search trees, inserting and
deleting in a hash table is more efficient, assuming rehashing stays infrequent.
The main disadvantage is needing a *good* hash function, though this is rarely
an issue in practice.

**Hash function requirements.**
- **Hard requirement:** equal keys must produce equal hash codes. Easy to get
  wrong — e.g., hashing on an object's memory address rather than its contents,
  or including profiling data in the hash.
- **Soft requirement:** the hash function should "spread" keys — hash codes for
  a subset of objects should be uniformly distributed across the array — and
  should be efficient to compute.

**A common mistake.** If a key already present in a hash table is mutated in
place, a subsequent lookup for that key will fail — even though it's still
physically in the table, because it now hashes to a different slot. **Rule of
thumb:** if you must update a key, remove it, update it, then re-add it. As a
general rule, avoid using mutable objects as keys.

## 2. Designing a hash function for strings

A good string hash function should:
- Examine **all** characters in the string.
- Produce a large range of values, and not let one character dominate (e.g.,
  naively casting characters to integers and multiplying them means a single
  `0` character zeroes out the whole hash code).
- Ideally be a **rolling hash**: if a character is removed from the front of the
  string and another appended to the end, the new hash code can be recomputed in
  `O(1)` time.

In [1]:
import functools

def string_hash(s, modulus):
    MULT = 997
    return functools.reduce(lambda v, c: (v * MULT + ord(c)) % modulus, s, 0)

print(string_hash("logarithmic", 10**9 + 7))
print(string_hash("algorithmic", 10**9 + 7))

579972068
916422259


A hash table is a good structure for representing a **dictionary** (a set of
strings). In some applications a **trie** — a tree that stores a dynamic set of
strings via a node's *position* in the tree, rather than a stored key — has
computational advantages instead.

## 3. Hash tables - Setup

Two examples: one shows an **application** that benefits from hash tables'
algorithmic advantages, the other shows the **design of a hashable class**.

### 3.1 An application of hash tables — grouping anagrams

**Problem.** Given a set of words, return the groups of anagrams among them
(each group must contain at least two words). For example, given `"debitcard"`,
`"elvis"`, `"silent"`, `"badcredit"`, `"lives"`, `"freedom"`, `"listen"`,
`"levis"`, `"money"`, there are three anagram groups; `"money"` belongs to none.

**Key idea.** Two words are anagrams iff sorting their characters yields the
same string. So a word's *sorted* form is a natural unique identifier ("key")
for its anagram group. We build a hash table mapping sorted-string → list of
original strings that sort to it.

In [2]:
import collections

def find_anagrams(dictionary):
    sorted_string_to_anagrams = collections.defaultdict(list)
    for s in dictionary:
        # Sorts the string, uses it as a key, and then appends the original
        # string as another value into the hash table.
        sorted_string_to_anagrams[''.join(sorted(s))].append(s)
    return [
        group for group in sorted_string_to_anagrams.values() if len(group) >= 2
    ]

words = ["debitcard", "elvis", "silent", "badcredit", "lives",
         "freedom", "listen", "levis", "money"]
print(find_anagrams(words))

[['debitcard', 'badcredit'], ['elvis', 'lives', 'levis'], ['silent', 'listen']]


**Complexity.** Sorting all `n` keys (each up to length `m`) costs
`O(n · m log m)`; the `n` hash table insertions add `O(n · m)`. Total:
`O(n · m log m)`.

### 3.2 Design of a hashable class — a `ContactList`

**Problem.** Represent a contact as a list of name strings. Two `ContactList`s
should be **equal** if they contain the same *set* of strings, regardless of
order or duplicate repetitions. To store these in a hash table (or a `set`), we
must define both `__eq__` and `__hash__` consistently — the hash must depend
only on the *set* of names, not their order or multiplicity.

Since Python's built-in `set` is mutable (and therefore unhashable), we hash a
**`frozenset`** of the names instead.

In [3]:
class ContactList:
    def __init__(self, names):
        '''names is a list of strings.'''
        self.names = names

    def __hash__(self):
        # Conceptually we want to hash the set of names. Since the set type is
        # mutable, it cannot be hashed. Therefore we use frozenset.
        return hash(frozenset(self.names))

    def __eq__(self, other):
        return set(self.names) == set(other.names)


def merge_contact_lists(contacts):
    '''contacts is a list of ContactList.'''
    return list(set(contacts))

In [4]:
# Demo: order and duplicates don't matter for equality/hashing
c1 = ContactList(["ana", "bob"])
c2 = ContactList(["bob", "ana", "ana"])   # same set, different order + a dup
c3 = ContactList(["cara", "dan"])

print("c1 == c2:", c1 == c2)
print("hash(c1) == hash(c2):", hash(c1) == hash(c2))
print("merged:", [sorted(set(c.names)) for c in merge_contact_lists([c1, c2, c3])])

c1 == c2: True
hash(c1) == hash(c2): True
merged: [['cara', 'dan'], ['ana', 'bob']]


**Complexity.** Computing the hash is `O(n)`, where `n` is the number of
strings in the contact list. Hash codes are often cached for performance — but
the cache must be invalidated whenever a field the hash function inspects is
updated.

## 4. Top tips for hash tables

- Hash tables give the best theoretical and real-world performance for lookup,
  insert, and delete — all `O(1)` on average. (A single insert can be `O(n)` in
  the worst case, if it triggers a resize.)
- Consider using a **hash code as a signature** to enhance performance, e.g., to
  quickly filter out candidates before a more expensive check.
- Consider a **precomputed lookup table** instead of boilerplate if/then logic
  for mappings — e.g., character → value, or character → character.
- When defining your own type for use in a hash table, understand the
  relationship between **logical equality and the fields the hash function must
  inspect**. Whenever `__eq__` is implemented, the matching `__hash__` must be
  implemented too — otherwise logically-equal objects can land in different
  buckets, causing lookups to spuriously fail even when the item is present.
- Sometimes you need a **multimap** (multiple values per key) or a
  **bi-directional map**. If the standard library doesn't provide it, use a
  `dict` of lists as values, or reach for a third-party library.

## 5. Know your hash table libraries

Python offers several hash table–based structures: **`set`**, **`dict`**,
**`collections.defaultdict`**, and **`collections.Counter`**. `set` stores only
keys; the other three store key-value pairs. None of them allow duplicate keys
(unlike `list`).

### 5.1 `dict` vs `collections.defaultdict`

Accessing a missing key in a plain `dict` raises a `KeyError`. A `defaultdict`
instead returns the default value for the type it was instantiated with.

In [5]:
d = {}
try:
    d['missing']
except KeyError as e:
    print("dict raised KeyError:", e)

dd = collections.defaultdict(list)
print("defaultdict returns default:", dd['missing'])   # [] — no KeyError

dict raised KeyError: 'missing'
defaultdict returns default: []


### 5.2 `collections.Counter`

`Counter` counts occurrences of keys and supports set-like arithmetic: `+`
(add counts), `-` (subtract, keeping only positive counts), `&` (intersection —
`min` of counts), `|` (union — `max` of counts).

In [6]:
c = collections.Counter(a=3, b=1)
d = collections.Counter(a=1, b=2)

print("c + d :", c + d)   # {'a': 4, 'b': 3}
print("c - d :", c - d)   # {'a': 2}          (b would be 1-2=-1, dropped)
print("c & d :", c & d)   # {'a': 1, 'b': 1}  (min per key)
print("c | d :", c | d)   # {'a': 3, 'b': 2}  (max per key)

c + d : Counter({'a': 4, 'b': 3})
c - d : Counter({'a': 2})
c & d : Counter({'a': 1, 'b': 1})
c | d : Counter({'a': 3, 'b': 2})


### 5.3 `set` operations

Key operations: `s.add(x)`, `s.remove(x)`, `s.discard(x)` (like `remove`, but no
error if absent), `x in s`, `s <= t` (is `s` a subset of `t`), and `s - t`
(elements in `s` not in `t`).

In [7]:
s = {1, 2, 3}
t = {1, 2, 3, 4, 5}

s.add(10)
s.discard(999)     # no-op, 999 isn't present — does NOT raise

print("s:", s)
print("3 in s:", 3 in s)
print("{1,2} <= t (subset):", {1, 2} <= t)
print("s - t:", s - t)

s: {10, 1, 2, 3}
3 in s: True
{1,2} <= t (subset): True
s - t: {10}


### 5.4 Iterating over key-value collections

Iterating over a `dict`/`defaultdict`/`Counter` directly yields **keys**. Use
`.items()` for key-value pairs and `.values()` for values only.

In [8]:
prices = {'apple': 1.5, 'banana': 0.5, 'cherry': 4.0}

print("iterating directly ->", [k for k in prices])       # keys
print("keys()             ->", list(prices.keys()))
print("values()           ->", list(prices.values()))
print("items()            ->", list(prices.items()))

iterating directly -> ['apple', 'banana', 'cherry']
keys()             -> ['apple', 'banana', 'cherry']
values()           -> [1.5, 0.5, 4.0]
items()            -> [('apple', 1.5), ('banana', 0.5), ('cherry', 4.0)]


### 5.5 Hashability

Not every type is **hashable** (usable as a `set` element or `dict` key).
**Mutable containers** (`list`, `dict`, `set`) are not hashable — this prevents
a client from mutating an object after insertion, which would otherwise strand
it in the wrong slot and break lookups. Use `frozenset` (as in `ContactList`
above) or `tuple` when you need a hashable, set-like or sequence-like key.

The built-in `hash()` function can greatly simplify implementing `__hash__` for
a user-defined class.

In [9]:
try:
    hash([1, 2, 3])   # list is mutable -> unhashable
except TypeError as e:
    print("list is unhashable:", e)

print("tuple is hashable:", hash((1, 2, 3)))
print("frozenset is hashable:", hash(frozenset([1, 2, 3])))

list is unhashable: unhashable type: 'list'
tuple is hashable: 529344067295497451
frozenset is hashable: -272375401224217160
